<a href="https://colab.research.google.com/github/michalis0/DataScience_and_MachineLearning/blob/master/Labs/10-explainable-ai/explainable-ai.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# Lab 10 · Explainable AI

## Learning goals

- Compute SHAP attributions and examine their direction and magnitude.
- Use LIME to explain individual text and tabular predictions.
- Explain how these local attributions relate to the model prediction.

## Before you start

In Colab, run the setup cell before the walkthrough. For local use, open this notebook from its lab folder so the relative data paths resolve. Run the walkthrough in order, then try the practice cells.

For a new environment, install the [lab requirements](../requirements.txt) with `python -m pip install -r Labs/requirements.txt` from the repository root.

In [ ]:
# In Colab, fetch the course data and switch to this lab folder.
from pathlib import Path
import os
import subprocess

try:
    from google import colab
except ImportError:
    colab = None

if colab is not None:
    course = Path("/content/dsml-course")
    lab = "Labs/10-explainable-ai"
    if not (course / ".git").is_dir():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
             "--branch", "master",
             "https://github.com/michalis0/DataScience_and_MachineLearning.git", str(course)],
            check=True,
        )
    if not (course / lab / "data").is_dir():
        subprocess.run(["git", "-C", str(course), "sparse-checkout", "add", lab], check=True)
    os.chdir(course / lab)
    %pip install -q shap lime

In [ ]:
import pandas as pd
import seaborn as sns
import shap
from sklearn.feature_extraction.text import TfidfVectorizer
from lime.lime_text import LimeTextExplainer
import matplotlib.pyplot as plt
from IPython.display import HTML

# Interpretability

So far we have used various models as black boxes, where an input goes in and an output goes out.

Interpretability allows us to understand what happens inside this blackbox.

<img src='https://assets.spe.org/dims4/default/f6b3d58/2147483647/strip/true/crop/696x392+0+0/resize/800x451!/quality/90/?url=http%3A%2F%2Fspe-brightspot.s3.us-east-2.amazonaws.com%2Fae%2F27%2F97e3d6195614a45bed36e7a965e2%2Fblackbox.jpg' width="500">

## Content

The goal of this walkthrough is to introduce explainability modules for Machine Learning models. Those allow us to better retrace the steps taken by the computer.

## Background <a class="anchor" id="Background"></a>

### Objective

AI is often perceived as a black box. This erodes trust and makes it harder for some industries to adopt it.

In order to mitigate this, XAI models (explainable AI) are being developped that allow us to retrace the steps performed by the computer.

### Examples of XAI models
User interface examples:


*   [Language level](http://deeptext.unice.fr/FLE/)
*   [Emotion classification](https://shap.readthedocs.io/en/latest/example_notebooks/text_examples/sentiment_analysis/Emotion%20classification%20multiclass%20example.html)



Various XAI libraries exist, the two most popular ones being `shap`and `lime`

## SHAP (SHapley Additive exPlanations)

**SHAP** is a popular library used for interpreting the output of machine learning models. It is based on the concept of Shapley values from cooperative game theory, which allocate the contribution of each feature to the model's predictions.

### How SHAP Works:
1. **Feature Contribution**: For each prediction, SHAP computes the contribution of each feature to the prediction by considering all possible feature combinations.
2. **Additive Feature Attribution**: The sum of SHAP values for all features equals the model’s prediction, ensuring consistency.

### Explain a text classification model

Use a small, labeled teaching dataset to see how masking individual words changes a model’s predicted probability.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

training_text = [
    "I am happy to be here", "This was a wonderful experience",
    "I enjoyed following the course", "The lesson was helpful and enjoyable",
    "I am sad that it ended", "This was a terrible experience",
    "I disliked the lecture", "The lesson was confusing and disappointing",
]
labels = [1, 1, 1, 1, 0, 0, 0, 0]
sentiment = make_pipeline(TfidfVectorizer(), LogisticRegression())
sentiment.fit(training_text, labels)

In [ ]:
def positive_probability(texts):
    return sentiment.predict_proba(list(texts))[:, 1]

In [ ]:
shap_explainer = shap.Explainer(positive_probability, shap.maskers.Text())

In [ ]:
shap_values = shap_explainer(["I am happy to be here although it is sad the course ended"])
shap.plots.text(shap_values)

### Practice

Plot a `shap` barplot that represents the weights of each word in a sentence for different emotions. You can use the documentation available [here](https://shap.readthedocs.io/en/latest/example_notebooks/text_examples/sentiment_analysis/Emotion%20classification%20multiclass%20example.html).

In [ ]:
# Your code here

## LIME
We will now create a similar model using `lime` explainability over youtube spam classifications. To do so we will train a model on the labeled commets of the videos from PSY, Shakira, LMFAO and Katy Perry, and then predict if some comments are spam or not.

### Classification
We will try to identify bellow the importance of each factor in the computers decision making.

We will apply it on two models, a `Random Forest Classifier` and a `Decision Tree`

In [ ]:
psy = pd.read_csv('data/Youtube01-Psy.csv')
perry = pd.read_csv('data/Youtube02-KatyPerry.csv')
LMFAO = pd.read_csv('data/Youtube03-LMFAO.csv')
Eminem = pd.read_csv('data/Youtube04-Eminem.csv')
Shakira = pd.read_csv('data/Youtube05-Shakira.csv')
full_train = pd.concat([psy, perry, LMFAO, Shakira, Eminem]).reset_index().drop(columns = ['index'])

In [ ]:
display(full_train.head(3))

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

text_train, text_test, y_train, y_test = train_test_split(
    full_train["CONTENT"], full_train["CLASS"],
    test_size=0.3, stratify=full_train["CLASS"], random_state=13,
)
tfidf = TfidfVectorizer(stop_words="english")
train_features = tfidf.fit_transform(text_train).toarray()
test_features = tfidf.transform(text_test).toarray()
X_train = pd.DataFrame(train_features, columns=tfidf.get_feature_names_out())
X_test = pd.DataFrame(test_features, columns=X_train.columns)

rf_clf = RandomForestClassifier(max_features=2, n_estimators=100, random_state=13)
rf_clf.fit(X_train, y_train)

def predict_fn_clf(values):
    return rf_clf.predict_proba(pd.DataFrame(values, columns=X_train.columns))

print(classification_report(y_test, rf_clf.predict(X_test)))

In [ ]:
# Import the LimeTabularExplainer module
from lime.lime_tabular import LimeTabularExplainer

# Get the class names
class_names = ['Not spam', 'Spam']

# Get the feature names
feature_names = list(X_train.columns)

# Fit the Explainer on the training data set using the LimeTabularExplainer
lime_explainer = LimeTabularExplainer(X_train.values, feature_names =
                                 feature_names,
                                 class_names = class_names,
                                 mode='classification', random_state=13)

In [ ]:
chosen_instance = X_test.iloc[0].to_numpy()
exp = lime_explainer.explain_instance(chosen_instance, predict_fn_clf, num_features=10)
display(pd.DataFrame(exp.as_list(), columns=["Feature", "Weight"]))

### Second example: Logistic regression

In the context of text classification, **sincere** and **insincere** often refer to whether a question (or text) is genuine and constructive or malicious, sarcastic, or disruptive. This distinction is commonly used in datasets like Quora Question Pairs or similar platforms where user-generated content is evaluated for quality or intent.



For this example we will analyse different tags on Quora posts

In [ ]:
# Paths to the split files
file_paths = [
    "data/Quora_pt1.csv",
    "data/Quora_pt2.csv",
    "data/Quora_pt3.csv"
]

# Read and combine all parts
train_df = pd.concat([pd.read_csv(path) for path in file_paths], ignore_index=True).dropna(subset=["question_text", "target"])
# Keep the in-class example small enough to run on a laptop.
train_df = train_df.groupby("target", group_keys=False).sample(frac=0.05, random_state=2018)

# Display the shape and first few rows
print("Train shape : ", train_df.shape)
display(train_df.head())

In [ ]:
## split to train and val (values to test)
train_df, val_df = train_test_split(train_df, test_size=0.1, random_state=2018)
val_df.reset_index(drop=True)

## vectorize to tf-idf vectors
tfidf_vc = TfidfVectorizer(min_df = 10, max_features = 100000, analyzer = "word", ngram_range = (1, 2), stop_words = 'english', lowercase = True)
train_vc = tfidf_vc.fit_transform(train_df["question_text"])
val_vc = tfidf_vc.transform(val_df["question_text"])

In [ ]:
## Training our model
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(C = 0.5, solver = "sag")
model = model.fit(train_vc, train_df.target)
val_pred = model.predict(val_vc)

## checking the accuracy of the model
from sklearn.metrics import accuracy_score
print("accuracy score = {}".format(accuracy_score(val_df.target, val_pred)))

Predictions and explaining one of the predictions:

Some good examples of divisive questions are indexed at : 15, 163, 226, 240, 306, and 979

In [ ]:
from sklearn.pipeline import make_pipeline
idx = val_df.index[979]

c = make_pipeline(tfidf_vc, model)
class_names = ["sincere", "insincere"]
explainer = LimeTextExplainer(class_names=class_names, random_state=2018)
exp = explainer.explain_instance(val_df["question_text"][idx], c.predict_proba, num_features = 10)

print("Question: \n", val_df["question_text"][idx])
print("Probability (Insincere) =", c.predict_proba([val_df["question_text"][idx]])[0, 1])
print("Probability (Sincere) =", c.predict_proba([val_df["question_text"][idx]])[0, 0])

In [ ]:
exp.as_list()

In [ ]:
display(HTML(exp.as_html(labels=(1,))))

In [ ]:
import collections

weights = collections.OrderedDict(exp.as_list())
lime_weights = pd.DataFrame({"words": list(weights.keys()), "weights": list(weights.values())})

sns.barplot(x = "words", y = "weights", data = lime_weights)
plt.xticks(rotation = 45)
plt.title("Sample {} features weights given by LIME".format(idx))
plt.show()